In [ ]:
!rm -rf ./src && git clone https://github.com/emiliodelgadouy/TesisV4_src src
from src.notebook import configure_notebook
configure_notebook(autoreload=True)
from src.notebook_api import *

Cloning into 'src'...


remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 48 (delta 1), reused 48 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 56.57 KiB | 616.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.


In [ ]:
GENERAL = {
    "RANDOM_SEED": 42,
    "REDUCED_DATASET": False,
    "USE_CLAHE": True,
    "POSITIVE_MODE": "mass",  # "mass": positivos = Mass==1 | "full": positivos = cualquier hallazgo (cls==1)
    "VALIDATION_SPLIT_RATIO": 0.20,
    "PROBABILITY_THRESHOLD": 0.5,  # referencia secundaria; reporte principal con umbral Youden (val)
    "NO_FINDING_TO_FINDING_RATIO": 3,
    "METRIC_TO_MAXIMIZE": "auc",
    "BATCH_SIZE": 32,
    # tf.data cache del pipeline procesado (simple/patch: pocos GB; full/abmil: poner False si falta RAM).
    "CACHE_DATASET": True,
    # Splits fijos (ids) en splits/dataset_splits.json. True: regenera y guarda; False: carga los fijos.
    "REGENERATE_SPLITS": False,
}
TRAINING = {
    "EPOCHS_FROZEN_BACKBONE": 10,  # produccion: 10
    "EPOCHS_PARTIAL_BACKBONE": 20,  # produccion: 20
    "EPOCHS_FULL_FINETUNE": 10,  # produccion: 10 (early stopping corta antes)
    "FOCAL_GAMMA": 2.0,
    "AGGRESSIVE_AUGMENTATION": True,
    "BACKBONE_TRAINABLE_FRACTION": 0.30,
    "EARLY_STOPPING_PATIENCE": 4,
    "REDUCE_LR_PATIENCE": 2,
}
MIL = {
    "BAG_KERAS_TILING": True,
    "BATCH_SIZE": 32,  # solo ABMIL; SIMPLE/PATCH/FULL usan GENERAL → BATCH_SIZE
    "ATTENTION_DIM": 128,
    "ATTENTION_GATED": True,
}
PATCH = {
    "POSITIVE": 1,
    "HARD_NEGATIVE": 0,
    "RANDOM_NEGATIVE": GENERAL["NO_FINDING_TO_FINDING_RATIO"],
    # Reescala M×M al canvas F×F (FULL) antes del crop S×S.
    "RESIZE_TO_BAG_CANVAS": True,
}
PATCH_HARDNEG = {
    "POSITIVE": 1,
    "HARD_NEGATIVE": GENERAL["NO_FINDING_TO_FINDING_RATIO"] / 2,
    "RANDOM_NEGATIVE": GENERAL["NO_FINDING_TO_FINDING_RATIO"] / 2,
    # True: crops en las G² posiciones de la grilla ABMIL; False: crop libre como PATCH.
    "ALIGN_TO_BAG_GRID": True,
}
COMET = {
    # Preferir env COMET_API_KEY / ~/.comet.config; no versionar la key.
    "API_KEY": "SzBk2RvzTzZCQMpLNBh35mg1q",
    "PROJECT_NAME": "tesis_v3",
    "N_SAMPLE_IMAGES": 8,
}
MODELS = [
    "customtiny",
    # "efficientnetb0",
    # "efficientnetv2b0",
    # backbones pesados — descomentar solo para corridas finales:
    # "chexnet",
    # "vgg19",
]
FULL = {
    "BAG_GRID": (3, 3),  # G: grilla compartida FULL/ABMIL; F default = G×S del backbone
    "BAG_CANVAS_MODE": "resize",  # cómo adaptar M×M al canvas F×F (resize | pad)
}
CONFIG = {
    "GENERAL": GENERAL,
    "TRAINING": TRAINING,
    "MIL": MIL,
    "COMET": COMET,
    "MODELS": MODELS,
    "PATCH": PATCH,
    "PATCH_HARDNEG": PATCH_HARDNEG,
    "FULL": FULL,
}
set_random_seeds(CONFIG)
login_comet(CONFIG)

COMET INFO: Valid Comet API Key saved in /root/.comet.config (set COMET_CONFIG to change where it is saved).


## Preparación del dataset

Carga los datos, elimina imágenes duplicadas y construye la etiqueta binaria utilizada durante el entrenamiento y la evaluación.

**Parámetros:**
- `GENERAL → REDUCED_DATASET`: utiliza una muestra reducida para realizar pruebas rápidas.
- `GENERAL → POSITIVE_MODE`: con `mass` considera positivas solo las masas; con `full`, cualquier hallazgo.
- `GENERAL → RANDOM_SEED`: fija la semilla para obtener resultados reproducibles.

In [ ]:
ds = prepare_dataset(CONFIG)

GET https://storage.googleapis.com/helen-data/square_images.tar.gz


Images downloaded
Extracting images...


Images extracted to /content/mammo/raw/images
GET https://storage.googleapis.com/helen-data/square_data.csv


CSV downloaded
GET https://storage.googleapis.com/helen-data/dataset_splits.json


Splits downloaded to /content/splits/dataset_splits.json


deduplicate_images: 1226 -> 1113 rows (113 duplicates removed)
TARGET_MODE=mass: 1113 positivos / 18232 negativos (19345 total)


## División del dataset

Divide los datos en conjuntos de entrenamiento, validación y prueba. Puede reutilizar la división guardada en `splits/dataset_splits.json` (se descarga de GCS junto al CSV e imágenes) o generar una nueva; al regenerarla también reduce la cantidad de negativos del conjunto de entrenamiento.

**Parámetros:**
- `GENERAL → REGENERATE_SPLITS`: genera y guarda una división nueva si es `True`; carga la existente si es `False`.
- `GENERAL → VALIDATION_SPLIT_RATIO`: define la proporción destinada a validación.
- `GENERAL → NO_FINDING_TO_FINDING_RATIO`: define la cantidad de negativos por cada positivo en entrenamiento.
- `GENERAL → RANDOM_SEED`: fija la semilla para obtener una división reproducible.

In [ ]:
train, val, test = get_dataset_splits(CONFIG, ds)

Splits cargados desde /content/splits/dataset_splits.json (train=2864, val=3097, test=3862, meta_keys=['includes_undersampled_train', 'n_test', 'n_train', 'n_val', 'no_finding_to_finding_ratio', 'positive_mode', 'random_seed', 'reduced_dataset', 'validation_split_ratio'])


## Configuración común (todos los modos)

**Parámetros compartidos:**
- `B` (`GENERAL → BATCH_SIZE`): tamaño del lote; en ABMIL se usa `MIL → BATCH_SIZE`
- `S` (`MODELS`): tamaño nativo del backbone (entrada de SIMPLE / tiles)
- `F` (`FULL → INPUT_SIZE`): tamaño de entrada de FULL; **`F > S`** (más resolución que SIMPLE)
- `G` (`FULL → BAG_GRID`): grilla compartida; `F` default = `G×S` si no hay `INPUT_SIZE`
- `FULL → BAG_CANVAS_MODE`: cómo adaptar la mamografía al canvas `F×F`
- `GENERAL → CACHE_DATASET`: cachea o no el dataset procesado (False en FULL/ABMIL si falta RAM)
- `M`: resolución original de la mamografía (`S ≤ F ≤ M` en el caso típico; `F` puede superar `M` si hay upsampling)
- `GENERAL → USE_CLAHE`: realce de contraste
- `GENERAL → METRIC_TO_MAXIMIZE`: métrica para guardar el mejor modelo (p. ej. `auc`)
- `GENERAL → PROBABILITY_THRESHOLD`: umbral de referencia (el reporte principal usa Youden)

**Etapas de entrenamiento** (`TRAINING`):

1. Backbone congelado: `EPOCHS_FROZEN_BACKBONE` épocas.
2. Fine-tune parcial del último `BACKBONE_TRAINABLE_FRACTION` del backbone: `EPOCHS_PARTIAL_BACKBONE` épocas.
3. Fine-tune completo: `EPOCHS_FULL_FINETUNE` épocas.

En todas se monitorea la métrica elegida, con detención temprana (`EARLY_STOPPING_PATIENCE`) y reducción de LR (`REDUCE_LR_PATIENCE`). También se controlan el aumento de datos (`AGGRESSIVE_AUGMENTATION`) y la pérdida focal (`FOCAL_GAMMA`).


## Entrenamiento SIMPLE

Entrena un clasificador convencional con la mamografía completa reducida a `S×S`, donde `S` es la entrada nativa o recomendada del backbone. Ese tamaño no siempre es una restricción matemática de la CNN, pero es la escala para la que se configuró la arquitectura y, cuando corresponde, la más cercana a su preentrenamiento. Conserva el contexto global de la mama y evita artefactos producidos por fronteras entre parches. Trabajar cerca de la resolución nativa del backbone reduce el cambio de escala respecto de los pesos preentrenados y facilita comparaciones entre los backbone .
Procesa la mamografía en una sola pasada, por lo que suele permitir batches más grandes y entrenamientos más rápidos.


**Limitaciones:**
- Siempre reescala la imagen completa a `S×S`. Cuando `S` es mucho menor que `M`, la compresión elimina detalle espacial y puede borrar o suavizar lesiones pequeñas y bordes finos.
- Un backbone pequeño puede combinar dos limitaciones diferentes: menor capacidad del modelo y menor resolución de entrada.
- No incorpora un mecanismo explícito de localización. Una predicción correcta no garantiza que el modelo haya utilizado la región clínicamente relevante.

**Pipeline:**

`[B, M, M, 3] → resize [B, S, S, 3] → augmentación → preprocess(BACKBONE) → BACKBONE → GAP → Dense(256) → Dropout(0.4) → logit [B, 1]`


In [ ]:
for backbone_name in CONFIG["MODELS"]:
    run_training_experiment(CONFIG, "simple", backbone_name, train, val, test)

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/trainer.py:212: UserWarning: Model doesn't support `jit_compile=True`. Proceeding with `jit_compile=False`.
  warnings.warn(
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.


COMET WARNING: Failed to log Google Colab notebook URL, reason: list index out of range


COMET INFO: Experiment is live on comet.com https://www.comet.com/emiliodelgadouy/tesis-v3/f8803f3f19bd47dbb417455af84bfafc



COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.


  Warm-up completado en 19.2s
Epoch 1/10


...

COMET INFO:     model-element       : 1 (107.91 KB)


COMET INFO:     notebook            : 1


COMET INFO:     os packages         : 1


COMET INFO:     source_code         : 1


COMET INFO: 


## Entrenamiento FULL

Entrena un clasificador con la mamografía completa, sin dividirla en regiones. SIMPLE comprime a `S×S`; FULL usa una entrada propia `F×F` (`FULL → INPUT_SIZE`), con **`F > S`**, o sea más píxeles y menos downsampling que SIMPLE. FULL no elimina necesariamente el resize: la imagen todavía se adapta de `M×M` a `F×F`, pero la pérdida de detalle es menor cuando `F` está más cerca de `M`.

**Ventajas:**
- Retiene más detalle espacial que SIMPLE y, por lo tanto, ofrece una mejor oportunidad de conservar hallazgos pequeños.
- Mantiene simultáneamente información local y contexto global, sin imponer cortes entre regiones.
- En backbones convolucionales con `GlobalAveragePooling`, aumentar la dimensión espacial normalmente no cambia la cantidad principal de parámetros; los filtros preentrenados siguen siendo compatibles por forma.
- Es una comparación útil para separar el beneficio de mayor resolución del beneficio específico de la atención o del aprendizaje por instancias.

**Limitaciones:**
- El costo de activaciones, memoria y cómputo crece aproximadamente con la cantidad de píxeles (`(F/S)²` respecto de SIMPLE); puede exigir reducir el batch y evitar cachear imágenes procesadas.
- Si `F` sigue siendo menor que `M`, todavía se pierde información por downsampling. Si es mayor, el upsampling aumenta el tamaño pero no crea detalle nuevo.
- Los pesos convolucionales son compatibles con la entrada grande, pero existe un cambio de distribución de escala: estructuras que durante el preentrenamiento ocupaban cierta fracción de la imagen pasan a ocupar otra. Esto modifica el campo receptivo efectivo y puede afectar especialmente estadísticas de Batch Normalization; es un posible desajuste de transferencia, no una incompatibilidad de dimensiones.
- Continúa usando una representación global con GAP, por lo que una lesión pequeña puede diluirse aun cuando haya más píxeles disponibles.
- Una entrada mayor no garantiza mejor rendimiento: el beneficio debe compensar la mayor dificultad de optimización y validarse para cada backbone.

**Pipeline:**

`[B, M, M, 3] → resize [B, F, F, 3] → augmentación → preprocess(BACKBONE) → BACKBONE → GAP → Dense(256) → Dropout(0.4) → logit [B, 1]`

**Parámetros:**
- `F` (`FULL → INPUT_SIZE`): tamaño de entrada de FULL; **mayor que** el `S` de SIMPLE. Si no se setea, `F = FULL → BAG_GRID × S` para alinear escalas entre modos.
- `FULL → BAG_GRID`: lado `G` de la grilla compartida con ABMIL/patch
- `FULL → BAG_CANVAS_MODE`: cómo se adapta `M×M` al canvas `F×F`


In [ ]:
for backbone_name in CONFIG["MODELS"]:
    full_model = run_training_experiment(CONFIG, "full", backbone_name, train, val, test, return_builder=True)

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/trainer.py:212: UserWarning: Model doesn't support `jit_compile=True`. Proceeding with `jit_compile=False`.
  warnings.warn(
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.


COMET WARNING: Failed to log Google Colab notebook URL, reason: list index out of range


COMET INFO: Experiment is live on comet.com https://www.comet.com/emiliodelgadouy/tesis-v3/5047bb845fd54335abb12bb51327c7e3



COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.


  Warm-up completado en 19.1s
Epoch 1/10


...

COMET INFO:     model-element       : 1 (119.32 KB)


COMET INFO:     notebook            : 1


COMET INFO:     os packages         : 1


COMET INFO:     source_code         : 1


COMET INFO: 


## Entrenamiento ABMIL

Entrena un modelo de aprendizaje por múltiples instancias. Parte del mismo canvas que FULL (`F×F`), pero lo divide en una grilla `G×G` de regiones de tamaño `S×S` (el de SIMPLE). `TimeDistributed` aplica el mismo encoder a cada región (pesos compartidos); `GatedAttentionPooling` las vuelve a unificar en un vector de bag y recién ahí clasifica.

**Ventajas:**
- Cada región llega al backbone a `S×S`, preservando más detalle local que comprimir toda la mamografía directamente a `S×S` como en SIMPLE.
- La atención aprende qué regiones contribuyen más a la etiqueta global sin requerir una ROI durante la inferencia.
- Los pesos compartidos permiten aplicar el mismo detector de patrones locales en toda la mama y controlar la cantidad de parámetros.
- Combina evidencia de varias regiones; puede capturar múltiples focos o señales distribuidas que un único crop perdería.
- Los pesos de atención ofrecen una herramienta de inspección regional, aunque no deben interpretarse automáticamente como una explicación causal.

**Limitaciones:**
- Ejecuta el backbone `G²` veces por mamografía. El costo de memoria y cómputo puede ser comparable o superior a FULL, según la implementación y el batch.
- La grilla fija introduce fronteras: una lesión puede quedar dividida entre tiles. Sin solapamiento, ninguna instancia ve necesariamente la lesión completa.
- El pooling por atención actual no recibe coordenadas posicionales explícitas; conoce las características de las regiones, pero pierde gran parte de sus relaciones espaciales una vez codificadas.
- La supervisión es débil: solo se conoce la etiqueta del bag. El modelo debe descubrir qué tiles son relevantes y puede concentrarse en correlaciones espurias.
- La atención puede ser inestable o difusa cuando hay muchos tiles similares, y un peso alto no prueba que esa región sea suficiente ni necesaria para la predicción.
- La mamografía completa todavía se adapta al canvas `F×F` (por defecto el mismo que FULL); la escala final depende de `FULL → BAG_GRID` y `FULL → BAG_CANVAS_MODE`.

**Pipeline:**

`[B, M, M, 3] → resize [B, F, F, 3] → augmentación → preprocess(BACKBONE) → tiling G×G → [B, G², S, S, 3] → TimeDistributed(BACKBONE+GAP+Dense(256)) → [B, G², D] → gated attention → [B, D] → Dropout(0.4) → logit [B, 1]`

**Parámetros:**
- `G` (`FULL → BAG_GRID`): lado de la grilla `(G, G)`; regiones = `G²`; canvas = el de FULL (`F×F`)
- `MIL → BATCH_SIZE`: tamaño del lote (bags); reemplaza a `GENERAL → BATCH_SIZE` en este modo
- `MIL → BAG_KERAS_TILING`: si es `True`, el tiling se hace dentro del modelo
- `FULL → BAG_CANVAS_MODE`: cómo se adapta la mamografía al canvas (compartido con FULL)
- `MIL → ATTENTION_DIM`: dimensión de la atención
- `MIL → ATTENTION_GATED`: atención con o sin compuerta
- `GENERAL → CACHE_DATASET`: cachea o no el dataset procesado (False en FULL/ABMIL si falta RAM)


In [ ]:
for backbone_name in CONFIG["MODELS"]:
    abmil_model = run_training_experiment(CONFIG, "abmil", backbone_name, train, val, test, return_builder=True)

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/trainer.py:212: UserWarning: Model doesn't support `jit_compile=True`. Proceeding with `jit_compile=False`.
  warnings.warn(
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.


COMET WARNING: Failed to log Google Colab notebook URL, reason: list index out of range


COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.


COMET INFO: Experiment is live on comet.com https://www.comet.com/emiliodelgadouy/tesis-v3/70dcdf686b6449d69341d75d75bf5e5e



  Warm-up completado en 21.0s
Epoch 1/10


...

COMET INFO:     model-element       : 1 (893.57 KB)


COMET INFO:     notebook            : 1


COMET INFO:     os packages         : 1


COMET INFO:     source_code         : 1


COMET INFO: 


## Entrenamiento PATCH

Clasifica un crop de tamaño `S×S` (como SIMPLE), en lugar de la mamografía completa. Opcionalmente reescala antes al canvas de FULL/ABMIL (`F×F`) para muestrear a la misma escala. Los crops cambian aleatoriamente en train; en val/test son deterministas.

**Ventajas:**
- Concentra la capacidad del backbone en morfología local y evita gastar la mayor parte de la resolución en tejido alejado del hallazgo.
- Mantiene un costo similar a SIMPLE por muestra, aun cuando el parche provenga de un canvas de mayor resolución.
- Permite controlar explícitamente el muestreo de regiones positivas y negativas, y generar variación espacial entre épocas.
- Produce un encoder local que puede reutilizarse como inicialización de un modelo por bags.
- Con `PATCH → RESIZE_TO_BAG_CANVAS=True`, puede aproximar la escala visual que tendrán posteriormente los tiles de ABMIL.

**Limitaciones:**
- La selección positiva usa la ROI anotada. Por eso las métricas de PATCH en validación/test son métricas de parches asistidos por ROI, no rendimiento end-to-end sobre una mamografía sin anotaciones.
- El crop pierde contexto global y relaciones con otras regiones; también puede cortar una lesión grande o desplazarla hacia el borde.
- El resultado depende fuertemente de la calidad de las coordenadas ROI y de la estrategia de muestreo. Una ROI inválida, incompleta o imprecisa puede introducir etiquetas ruidosas.
- Positivos centrados en ROI y negativos provenientes de mamografías sin hallazgos pueden crear una tarea demasiado fácil o permitir atajos de distribución en lugar de aprender la lesión.
- Si la escala usada para recortar no coincide con la escala de un consumidor posterior, el encoder puede sufrir un cambio de dominio al transferirse.

**Pipeline:**

`[B, M, M, 3] → resize opcional [B, F, F, 3] → crop [B, S, S, 3] → augmentación → preprocess(BACKBONE) → BACKBONE → GAP → Dense(256) → Dropout(0.4) → logit [B, 1]`

**Parámetros:**
- `F` / canvas FULL: con `PATCH → RESIZE_TO_BAG_CANVAS=True`, canvas previo al crop = `F×F` (el de FULL/ABMIL)
- `PATCH`: proporciones de regiones positivas, negativas difíciles y negativas aleatorias
- `PATCH → RESIZE_TO_BAG_CANVAS`: activa el resize al canvas antes del crop


In [ ]:
for backbone_name in CONFIG["MODELS"]:
    run_training_experiment(CONFIG, "patch", backbone_name, train, val, test)

resample_train_for_patch: {'POSITIVE': 716, 'HARD_NEGATIVE': 0, 'RANDOM_NEGATIVE': 2148, 'TOTAL': 2864, 'FRAC_NEG': 0.75}


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/trainer.py:212: UserWarning: Model doesn't support `jit_compile=True`. Proceeding with `jit_compile=False`.
  warnings.warn(
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.


COMET WARNING: Failed to log Google Colab notebook URL, reason: list index out of range


COMET INFO: Experiment is live on comet.com https://www.comet.com/emiliodelgadouy/tesis-v3/69133af136a345328cac36f61781cf20



COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.


...

COMET INFO:     model-element       : 1 (119.32 KB)


COMET INFO:     notebook            : 1


COMET INFO:     os packages         : 1


COMET INFO:     source_code         : 1


COMET INFO: 


## Entrenamiento PATCH HARDNEG

Variante de PATCH que incorpora negativos difíciles. Entrena el encoder local que luego se transfiere a ABMIL en la celda siguiente.

**Ventajas:**
- Agrega parches negativos tomados fuera de la ROI en mamografías positivas. Esto reduce el atajo de distinguir simplemente entre tejido de una imagen con hallazgo y tejido de una imagen `No Finding`.
- Obliga al encoder a separar la lesión de tejido difícil del mismo paciente, adquisición y contexto mamográfico.
- Puede alinear los crops con la grilla ABMIL para disminuir el cambio de escala y geometría antes de la transferencia.

**Limitaciones:**
- Asume que lo situado fuera de la ROI es negativo. Hallazgos secundarios no anotados o ROIs incompletas pueden producir falsos negativos.
- Reutilizar una mamografía positiva como muestra positiva y negativa genera observaciones correlacionadas dentro de train. No es leakage entre splits, pero modifica el balance efectivo y debe considerarse al calibrar probabilidades.
- Sigue dependiendo de anotaciones ROI y sus métricas siguen siendo asistidas por ROI.
- Si los crops son libres y ABMIL usa tiles fijos, puede persistir una diferencia geométrica entre preentrenamiento y transferencia.

**Pipeline:**

`[B, M, M, 3] → resize opcional [B, F, F, 3] → crop [B, S, S, 3] → augmentación → preprocess(BACKBONE) → BACKBONE → GAP → Dense(256) → Dropout(0.4) → logit [B, 1]`

**Parámetros:**
- `PATCH_HARDNEG`: proporciones de positivas, negativas difíciles y negativas aleatorias (a diferencia de `PATCH`)
- `PATCH → RESIZE_TO_BAG_CANVAS`: usa el canvas de FULL/ABMIL antes del crop
- `PATCH_HARDNEG → ALIGN_TO_BAG_GRID`: si es `True`, los crops se restringen a las `G²` posiciones de la grilla ABMIL; si es `False`, son libres como en PATCH


## Entrenamiento ABMIL (desde PATCH HARDNEG)

Transfiere el encoder de PATCH HARDNEG a ABMIL. La salida global del bag se inicializa nueva.

**Ventajas:**
- Inicializa ABMIL con un backbone y una proyección densa que ya aprendieron a distinguir patrones locales de lesión frente a tejido difícil.
- Congelar inicialmente el encoder permite adaptar la agregación sin destruir de inmediato la representación local preentrenada.
- La inferencia ABMIL final procesa el bag completo y no necesita ROI, aunque la inicialización sí haya usado anotaciones.

**Limitaciones:**
- Puede haber transferencia negativa incluso con crops alineados: la supervisión patch sigue siendo más fuerte que la etiqueta global del bag.
- Es un procedimiento más costoso porque requiere entrenar, seleccionar y transferir dos modelos consecutivos.
- Usa supervisión ROI adicional durante el preentrenamiento. Una comparación con ABMIL puro debe explicitar que no ambos métodos reciben la misma cantidad de supervisión.

**Pipeline:**

`[B, M, M, 3] → resize [B, F, F, 3] → tiling G×G → encoder patch por tile → embeddings → atención ABMIL → embedding de bag → salida global nueva`

**Parámetros:**
- `FULL → BAG_GRID` / `BAG_CANVAS_MODE` y el resto de `MIL` (atención, batch) es el mismo que en ABMIL
- La celda entrena `patch_hardneg` y transfiere a `abmil_patch_hardneg`


In [ ]:
for backbone_name in CONFIG["MODELS"]:
    model = run_training_experiment(CONFIG, "patch_hardneg", backbone_name, train, val, test, return_builder=True)
    run_training_experiment(CONFIG, "abmil_patch_hardneg", backbone_name, train, val, test, pretrained_builder=model)

resample_train_for_patch: {'POSITIVE': 716, 'HARD_NEGATIVE': 1074, 'RANDOM_NEGATIVE': 1074, 'TOTAL': 2864, 'FRAC_NEG': 0.75}


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/trainer.py:212: UserWarning: Model doesn't support `jit_compile=True`. Proceeding with `jit_compile=False`.
  warnings.warn(
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.


COMET WARNING: Failed to log Google Colab notebook URL, reason: list index out of range


COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.


COMET INFO: Experiment is live on comet.com https://www.comet.com/emiliodelgadouy/tesis-v3/fd0080fd5c3b4b4f8ad356c5ac86dddd



...

COMET INFO:     model-element       : 1 (859.48 KB)


COMET INFO:     notebook            : 1


COMET INFO:     os packages         : 1


COMET INFO:     source_code         : 1


COMET INFO: 
